In [ ]:
pip install sentence-transformers

In [ ]:
"""
Tampa Zoning Code Embedding Pipeline v3 — Multi-Backend
========================================================
Extends v2 with configurable embedding backends so that
different text representation approaches can be compared
in a single run or swapped via the BACKEND config variable.

Backends supported:
  'tfidf'    — TF-IDF (8k terms, bigrams) + Truncated SVD
               No extra dependencies. Fast. Good baseline.

  'minilm'   — all-MiniLM-L6-v2  (sentence-transformers)
               384d → 32d PCA. Current v2 default.

  'mpnet'    — all-mpnet-base-v2  (sentence-transformers)
               768d → 32d PCA. Larger, richer representations.
               Expect ~3× slower encoding than MiniLM.

  'multi-qa' — multi-qa-mpnet-base-dot-v1  (sentence-transformers)
               768d → 32d PCA. Fine-tuned on Q&A pairs;
               better at matching regulatory concepts to
               factual queries. Experimental for zoning text.

  'openai'   — text-embedding-3-small via OpenAI API.
               Requires OPENAI_API_KEY env variable.
               3072d → 32d PCA. Highest quality, costs ~$0.01
               for 56 zone class corpora.

  'compare'  — Runs all available backends and prints a
               side-by-side similarity diagnostic and
               classifier comparison table.

Key change from v2:
  The corpus building and pedestrian scoring stages are
  IDENTICAL to v2. Only Stage 2 (embedding creation) changes.
  All outputs from Stage 1 (zone_text_corpora.csv,
  zone_ped_scores_v2.csv) are backwards-compatible.

Outputs:
  zone_text_corpora.csv           — same as v2
  zone_ped_scores_v2.csv          — same as v2
  zone_embeddings.csv             — embeddings from selected backend
  zone_embeddings_<backend>.csv   — per-backend copy when BACKEND='compare'
  fig_embedding_space_v3.png      — 2D PCA of embedding space
  fig_backend_comparison.png      — similarity + accuracy comparison (compare mode)
  embedding_classifier_results_v3.csv
"""

import re, os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.base import clone
from sklearn.decomposition import TruncatedSVD, PCA
from sklearn.discriminant_analysis import (LinearDiscriminantAnalysis,
                                            QuadraticDiscriminantAnalysis)
from sklearn.ensemble import (AdaBoostClassifier, GradientBoostingClassifier,
                               RandomForestClassifier)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, log_loss
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

# ─────────────────────────────────────────────────────────────
# CONFIGURATION
# ─────────────────────────────────────────────────────────────
# 'compare' runs all 7 backends and prints side-by-side diagnostics.
# Backend choice materially affects zone rankings; comparison is the contribution.
BACKEND = 'compare'   # 'tfidf' | 'minilm' | 'mpnet' | 'multi-qa' | 'openai' | 'compare'

CODE_FILES = [
    'raw_data/all_zoncode/Chapter_27___ZONING_AND_LAND_DEVELOPMENT.txt',
    'raw_data/all_zoncode/Chapter_26___UTILITIES.txt',
    'raw_data/all_zoncode/Chapter_22___STREETS_AND_SIDEWALKS.txt',
    'raw_data/all_zoncode/Chapter_17_5___AFFORDABLE_HOUSING__SUSTAINABILITY__AND_CONCURRENCY_MANAGEMENT_SYSTEM.txt',
    'raw_data/all_zoncode/Chapter_11___FIRE_PREVENTION_AND_PROTECTION.txt',
    'raw_data/all_zoncode/Chapter_6___BUSINESS_REGULATION.txt',
    'raw_data/all_zoncode/Chapter_1___GENERAL_PROVISIONS.txt',
    'raw_data/HILLSBOROUGH_COUNTY___LAND_DEVELOPMENT_CODE.txt',
]
TRIPS_PATH   = 'tampatrips_1.csv'
ZON_CSV_PATH = 'processed_data/tpazoning_clean.csv'
FEAT_PATH    = 'zone_features_combined.csv'

# 32 PCA components standardizes across backends (MiniLM=384d, OpenAI=3072d native);
# retains 97-99% variance for all backends while making DML input dimensions identical.
EMBEDDING_DIM  = 32    # output dimensions fed to classifiers
RANDOM_STATE   = 42

# ─────────────────────────────────────────────────────────────
# ZONE DEFINITIONS  (identical to v2)
# ─────────────────────────────────────────────────────────────
ALL_ZONES = [
    'RS-150','RS-100','RS-75','RS-60','RS-50',
    'RM-12','RM-16','RM-18','RM-24','RM-35','RM-50','RM-75',
    'RO-1','RO','OP-1','OP','CN','CG','CI','IG','IH',
    'PD-A','PD',
    'CBD-1','CBD-2',
    'SH-RS-A','SH-RS','SH-RM','SH-RO','SH-CN','SH-CG','SH-CI','SH-PD',
    'NMU-35','NMU-24','NMU-16',
    'M-AP-4','M-AP-3','M-AP-2','M-AP-1',
    'YC-9','YC-8','YC-7','YC-6','YC-5','YC-4','YC-3','YC-2','YC-1',
    'CD-3','CD-2','CD-1',
    'CU','UC','AS-1','RM-24/18',
]

RESIDENTIAL = [
    'RS-150','RS-100','RS-75','RS-60','RS-50',
    'RM-12','RM-16','RM-18','RM-24','RM-35','RM-50','RM-75',
    'RO','RO-1','SH-RS','SH-RS-A','SH-RM','SH-RO','SH-PD',
    'YC-2','YC-4','YC-8','YC-9',
]
NON_RESIDENTIAL = [
    'CG','CI','CN','OP','OP-1','IG','IH','CBD-1','CBD-2',
    'CD-1','CD-2','CD-3','NMU-35','NMU-24','NMU-16',
    'SH-CG','SH-CI','SH-CN','YC-1','YC-3','YC-5','YC-6','YC-7',
]
ZONE_CONTEXT = {z: ('Residential'     if z in RESIDENTIAL else
                    'Non-Residential'  if z in NON_RESIDENTIAL else 'Other/PD')
                for z in ALL_ZONES}

# ─────────────────────────────────────────────────────────────
# STAGE 1 — CORPUS BUILDING  (identical to v2, reproduced here
#           so the script is self-contained)
# ─────────────────────────────────────────────────────────────

# --- Section definitions (same as v2) ---
ZONE_SECTIONS = {
    'RS-150':['156','232','283'],      'RS-100':['156','232','283'],
    'RS-75': ['156','232','283'],      'RS-60': ['156','232','283'],
    'RS-50': ['156','232','283'],
    'RM-12': ['156','232','283'],      'RM-16': ['156','232','283'],
    'RM-18': ['156','232','283'],      'RM-24': ['156','232','283'],
    'RM-35': ['156','232','283'],      'RM-50': ['156','232','283'],
    'RM-75': ['156','232','283'],
    'RO':    ['156','164','232','233','283'],
    'RO-1':  ['156','164','232','233','283'],
    'OP':    ['156','232','233','283'], 'OP-1': ['156','232','233','283'],
    'CN':    ['156','164','232','233','283'],
    'CG':    ['156','232','233','283'], 'CI':   ['156','232','233','283'],
    'IG':    ['156','283'],             'IH':   ['156','283'],
    'PD':    ['227','283'],             'PD-A': ['227','283'],
    'CBD-1': ['181','181.1','181.2','181.3','181.4','181.5',
              '182','183','184','184.1','185','185.1','185.2','185.3',
              '206','283'],
    'CBD-2': ['181','181.1','181.2','181.3','181.4','181.5',
              '182','183','184','184.1','185','185.1','185.2','185.3',
              '206','283'],
    'SH-RS': ['211.2','211.2.1','211.2.2','211.2.4','211.2.5','211.8','283'],
    'SH-RS-A':['211.2','211.2.1','211.2.2','211.2.4','211.2.5','211.8','283'],
    'SH-RM': ['211.2','211.2.3','211.2.5','211.6','211.8','283'],
    'SH-RO': ['211.2','211.2.4','211.8','283'],
    'SH-CN': ['211.2','211.2.5','211.8','283'],
    'SH-CG': ['211.2','211.6','211.8','283'],
    'SH-CI': ['211.2','211.6','211.8','283'],
    'SH-PD': ['211.2','211.8','283'],
    'NMU-35':['212.2','212.3','212.4','212.6','283'],
    'NMU-24':['212.2','212.3','212.4','212.6','283'],
    'NMU-16':['212.2','212.3','212.4','212.6','283'],
    'M-AP-1':['171','283'], 'M-AP-2': ['171','283'],
    'M-AP-3':['171','283'], 'M-AP-4': ['171','283'],
    'YC-1':  ['197','198','283'], 'YC-2': ['197','198','283'],
    'YC-3':  ['197','198','283'], 'YC-4': ['197','198','283'],
    'YC-5':  ['197','198','283'], 'YC-6': ['197','198','283'],
    'YC-7':  ['197','198','283'], 'YC-8': ['197','198','283'],
    'YC-9':  ['197','198','283'],
    'CD-1':  ['197','199','200','201','201.1','202','203','204','205','206','283'],
    'CD-2':  ['197','199','200','201','201.1','202','203','204','205','206','283'],
    'CD-3':  ['197','199','200','201','201.1','202','203','204','205','206','283'],
    'CU':    ['283'], 'UC': ['283'], 'AS-1': ['283'], 'RM-24/18': ['156','232','283'],
}
UNIVERSAL_SECTIONS = ['283.16','290.7','155.3.4','283','282','284']
OVERLAY_SECTIONS = {
    'CG':  ['236','238','240','241','243','244'],
    'CI':  ['238','240','241','243'],
    'CN':  ['236','240','241','244'],
    'RM-24':['241','244'], 'RM-35':['241','244'],
    'CBD-1':['236','238','243','244'],
    'CBD-2':['236','238','243','244'],
}
SHARED_SECTIONS = {
    '211.2','211.2.1','211.2.2','211.2.3','211.2.4','211.2.5',
    '211.6','211.7','211.8','212.2','212.4','212.6','197','198','171',
}
NMU_TABLE_SECTION = '212.3'

LABELED_DIM_TEXT = {
    'RS-150': 'RS-150 zoning: min lot area 15000 sqft, lot width 100 ft, front setback 30 ft, max height 35 ft. Large-lot single-family residential.',
    'RS-100': 'RS-100 zoning: min lot area 10000 sqft, lot width 100 ft, front setback 25 ft, max height 35 ft. Single-family residential.',
    'RS-75':  'RS-75 zoning: min lot area 7500 sqft, lot width 75 ft, front setback 25 ft, max height 35 ft. Single-family residential.',
    'RS-60':  'RS-60 zoning: min lot area 6000 sqft, lot width 60 ft, front setback 25 ft, side setback 7 ft, rear setback 7 ft, max height 35 ft. Single-family residential dominant Tampa zone.',
    'RS-50':  'RS-50 zoning: min lot area 5000 sqft, lot width 50 ft, front setback 20 ft, side setback 7 ft, rear setback 7 ft, max height 35 ft. Smaller single-family residential lots.',
    'RM-12':  'RM-12 zoning: multi-family residential up to 12 units per acre, lot width 50 ft, front setback 25 ft, max height 35 ft.',
    'RM-16':  'RM-16 zoning: multi-family residential up to 16 units per acre, lot width 50 ft, front setback 25 ft, max height 35 ft.',
    'RM-18':  'RM-18 zoning: multi-family residential up to 18 units per acre, lot width 50 ft, front setback 25 ft, max height 35 ft.',
    'RM-24':  'RM-24 zoning: multi-family residential up to 24 units per acre, lot width 50 ft, front setback 25 ft, max height 60 ft. Medium density apartments and condominiums.',
    'RM-35':  'RM-35 zoning: multi-family residential up to 35 units per acre, lot width 50 ft, front setback 25 ft, max height 120 ft. Higher density multi-family residential.',
    'RM-50':  'RM-50 zoning: multi-family residential up to 50 units per acre, lot width 50 ft, front setback 25 ft, max height 200 ft. High-rise residential.',
    'RM-75':  'RM-75 zoning: multi-family residential up to 75 units per acre, lot width 50 ft, front setback 25 ft, no maximum height.',
    'RO':     'RO zoning: residential office district, lot width 50 ft, front setback 25 ft, max height 35 ft. Low-intensity office uses compatible with residential neighborhoods.',
    'RO-1':   'RO-1 zoning: residential office district higher intensity, lot width 50 ft, front setback 25 ft, max height 35 ft.',
    'OP':     'OP zoning: office park district, lot width 60 ft, front setback 25 ft, max height 60 ft. Professional office campus development.',
    'OP-1':   'OP-1 zoning: office park high-intensity, lot width 60 ft, front setback 20 ft, max height 200 ft. High-rise office.',
    'CN':     'CN zoning: commercial neighborhood district, lot width 60 ft, front setback 20 ft, max height 35 ft. Neighborhood-serving retail and services.',
    'CG':     'CG zoning: commercial general district, lot width 75 ft, front setback 10 ft, max height 45 ft. General commercial uses, auto-oriented.',
    'CI':     'CI zoning: commercial intensive district, lot width 100 ft, front setback 10 ft, no side setback, max height 45 ft. Intensive commercial including auto dealers and warehousing.',
    'IG':     'IG zoning: industrial general district, lot width 50 ft, front setback 10 ft, max height 60 ft. Light and general industrial.',
    'IH':     'IH zoning: industrial heavy district, lot width 50 ft, front setback 10 ft, no maximum height. Heavy manufacturing, storage, processing.',
    'CBD-1':  'CBD-1 zoning: central business district primary, zero front setback, zero side setback, zero rear setback, no maximum height. Downtown Tampa urban core.',
    'CBD-2':  'CBD-2 zoning: central business district secondary, zero setbacks, no maximum height. Downtown Tampa fringe, mixed uses.',
    'NMU-16': 'NMU-16 zoning: neighborhood mixed use, build-to line 15-20 ft front, zero side setback, max height 35 ft, transparency 35 percent minimum, no minimum parking.',
    'NMU-24': 'NMU-24 zoning: neighborhood mixed use, build-to line 15-20 ft front, zero side setback, max height 60 ft, transparency 50 percent minimum, no minimum parking.',
    'NMU-35': 'NMU-35 zoning: neighborhood mixed use, build-to line 15-20 ft front, zero side setback, max height 85 ft, transparency 65 percent minimum, no minimum parking.',
    'SH-RS':  'SH-RS zoning: Seminole Heights single-family residential subdistrict, lot width 50 ft, front setback 20 ft, max height 35 ft.',
    'SH-RS-A':'SH-RS-A zoning: Seminole Heights single-family attached subdistrict, lot width 50 ft, front setback 20 ft, max height 35 ft.',
    'SH-RM':  'SH-RM zoning: Seminole Heights multi-family residential subdistrict, lot width 50 ft, front setback 15 ft, max height 45 ft.',
    'SH-RO':  'SH-RO zoning: Seminole Heights residential-office subdistrict, lot width 50 ft, front setback 15 ft, max height 35 ft.',
    'SH-CN':  'SH-CN zoning: Seminole Heights commercial neighborhood subdistrict, lot width 50 ft, front setback 10 ft, no side setback, max height 35 ft.',
    'SH-CG':  'SH-CG zoning: Seminole Heights commercial general subdistrict, lot width 50 ft, front setback 10 ft, no side setback, max height 45 ft.',
    'SH-CI':  'SH-CI zoning: Seminole Heights commercial intensive subdistrict, lot width 50 ft, front setback 10 ft, no side setback, max height 45 ft.',
    'SH-PD':  'SH-PD zoning: Seminole Heights planned development subdistrict, custom development standards.',
    'M-AP-1': 'M-AP-1 zoning: MacDill Air Force Base airport district subzone 1, restricted development near active runway.',
    'M-AP-2': 'M-AP-2 zoning: MacDill airport district subzone 2, limited compatible uses near airfield.',
    'M-AP-3': 'M-AP-3 zoning: MacDill airport district subzone 3, transitional compatible uses.',
    'M-AP-4': 'M-AP-4 zoning: MacDill airport district subzone 4, outer compatible zone.',
    'CU':     'CU zoning: community use district, civic and institutional uses including parks schools government buildings.',
    'UC':     'UC zoning: urban community district, mixed civic institutional and residential uses in urban setting.',
    'AS-1':   'AS-1 zoning: airport support district, aviation-compatible uses near Tampa International Airport.',
}


def extract_zone_specific_paragraphs(section_body, zone_code, window=600):
    parts = []
    for m in re.finditer(re.escape(zone_code), section_body, re.IGNORECASE):
        start = max(0, m.start() - 80)
        end   = min(len(section_body), m.end() + window)
        parts.append(section_body[start:end])
    return ' '.join(parts)


def extract_nmu_block(section_body, zone_code):
    nmu_order = ['NMU-16', 'NMU-24', 'NMU-35']
    if zone_code not in nmu_order:
        return ''
    idx   = nmu_order.index(zone_code)
    start = section_body.find(zone_code)
    if start < 0:
        return ''
    if idx + 1 < len(nmu_order):
        end = section_body.find(nmu_order[idx + 1])
        if end < 0:
            end = len(section_body)
    else:
        end = len(section_body)
    return section_body[start:end]


def clean_text(t):
    t = re.sub(r'\t', ' ', t)
    t = re.sub(r'\s+', ' ', t)
    return t.strip()


# ─────────────────────────────────────────────────────────────
# LOAD AND PARSE CODE FILES
# ─────────────────────────────────────────────────────────────
print('=' * 60)
print('STAGE 1 — LOADING AND PARSING CODE FILES')
print('=' * 60)

sec_pat      = re.compile(r'Sec\.\s+27-([\d\w\.\-]+)\.', re.MULTILINE)
all_sections = {}
all_lines    = []

for fname in CODE_FILES:
    if not os.path.exists(fname):
        print(f'  SKIPPED (not found): {fname}')
        continue
    with open(fname, encoding='utf-8-sig') as f:
        raw = f.read()
    text = raw.replace('\r\n', '\n').replace('\r', '\n')
    all_lines.extend(text.split('\n'))
    splits = sec_pat.split(text)
    for i in range(1, len(splits), 2):
        sec_num = splits[i]
        body    = splits[i + 1] if i + 1 < len(splits) else ''
        if sec_num not in all_sections:
            all_sections[sec_num] = body
        else:
            all_sections[sec_num] += '\n' + body
    print(f'  {fname.split("/")[-1][:55]}')

print(f'\nTotal sections: {len(all_sections):,}  |  Lines: {len(all_lines):,}')

# Zone-name line index
zone_line_index = {z: [] for z in ALL_ZONES}
for line in all_lines:
    stripped = line.strip()
    if len(stripped) < 25:
        continue
    for z in ALL_ZONES:
        if z in line and re.search(rf'\b{re.escape(z)}\b', line, re.IGNORECASE):
            zone_line_index[z].append(stripped)

# Build corpora (v2 logic)
print('\nBuilding zone corpora...')
corpora = {}
for zone in ALL_ZONES:
    parts = []
    if zone in LABELED_DIM_TEXT:
        parts.append(LABELED_DIM_TEXT[zone])
    for s in ZONE_SECTIONS.get(zone, []):
        body = all_sections.get(s, '')
        if not body:
            continue
        if s == NMU_TABLE_SECTION:
            block = extract_nmu_block(body, zone)
            if block:
                parts.append(block)
        elif s in SHARED_SECTIONS:
            passage = extract_zone_specific_paragraphs(body, zone)
            parts.append(passage if passage else body[:2000])
        else:
            parts.append(body[:5000])
    for s in OVERLAY_SECTIONS.get(zone, []):
        body = all_sections.get(s, '')
        if body:
            parts.append(body[:2000])
    for s in UNIVERSAL_SECTIONS:
        body = all_sections.get(s, '')
        if body:
            parts.append(body[:1000])
    parts.extend(zone_line_index[zone][:80])
    corpora[zone] = clean_text(' '.join(parts))

# Save corpus (identical to v2 output)
corpus_df = pd.DataFrame([
    {'zone_class': z, 'context': ZONE_CONTEXT[z],
     'corpus_chars': len(corpora[z]), 'corpus': corpora[z]}
    for z in ALL_ZONES
])
corpus_df.to_csv('zone_text_corpora.csv', index=False)
print(f'Saved: zone_text_corpora.csv  ({len(corpus_df)} zones)')

zone_texts = [corpora[z] for z in ALL_ZONES]

# ─────────────────────────────────────────────────────────────
# STAGE 2 — EMBEDDING BACKENDS
# ─────────────────────────────────────────────────────────────

def embed_tfidf(texts, n_dims):
    """TF-IDF (8k terms, bigrams) + Truncated SVD.
    Non-neural baseline: if comparable to neural, embeddings aren't adding
    much beyond bag-of-words."""
    tfidf = TfidfVectorizer(max_features=8000, ngram_range=(1, 2),
                            min_df=2, sublinear_tf=True)
    mat   = tfidf.fit_transform(texts)
    n     = min(n_dims, mat.shape[0] - 1, mat.shape[1] - 1)
    svd   = TruncatedSVD(n_components=n, random_state=RANDOM_STATE)
    emb   = svd.fit_transform(mat)
    var   = svd.explained_variance_ratio_.sum()
    return emb, f'TF-IDF (8k) + SVD ({n}d, var={var:.1%})'


def embed_sentence_transformer(texts, model_name, n_dims):
    """Generic sentence-transformer backend with PCA reduction."""
    from sentence_transformers import SentenceTransformer
    model = SentenceTransformer(model_name)
    raw   = model.encode(texts, show_progress_bar=True, batch_size=8,
                         # L2 normalization enables cosine similarity comparisons across zones.
                         normalize_embeddings=True)
    # PCA fit on all 56 zones: this is feature construction, not prediction --
    # all zone text is observed, so there is no train/test leakage concern.
    pca   = PCA(n_components=n_dims, random_state=RANDOM_STATE)
    emb   = pca.fit_transform(raw)
    var   = pca.explained_variance_ratio_.sum()
    return emb, f'{model_name} + PCA ({n_dims}d, var={var:.1%})'


def embed_openai(texts, n_dims):
    """OpenAI text-embedding-3-small via API. Requires OPENAI_API_KEY."""
    import openai, time
    client = openai.OpenAI()           # reads OPENAI_API_KEY from env
    raw_vecs = []
    print(f'  Calling OpenAI API for {len(texts)} zone corpora...')
    # Truncate to 8192 tokens (approx 32k chars) per text
    truncated = [t[:32000] for t in texts]
    resp = client.embeddings.create(
        model='text-embedding-3-small',
        input=truncated,
    )
    raw_vecs = np.array([d.embedding for d in resp.data])
    pca = PCA(n_components=n_dims, random_state=RANDOM_STATE)
    emb = pca.fit_transform(raw_vecs)
    var = pca.explained_variance_ratio_.sum()
    print(f'  Raw dims: {raw_vecs.shape[1]} → PCA {n_dims}d  (var={var:.1%})')
    return emb, f'OpenAI text-embedding-3-small + PCA ({n_dims}d)'


BACKEND_DEFS = {
    'tfidf':    lambda: embed_tfidf(zone_texts, EMBEDDING_DIM),
    'minilm':   lambda: embed_sentence_transformer(
                    zone_texts, 'all-MiniLM-L6-v2', EMBEDDING_DIM),
    'mpnet':    lambda: embed_sentence_transformer(
                    zone_texts, 'all-mpnet-base-v2', EMBEDDING_DIM),
    'multi-qa': lambda: embed_sentence_transformer(
                    zone_texts, 'multi-qa-mpnet-base-dot-v1', EMBEDDING_DIM),
    'openai':   lambda: embed_openai(zone_texts, EMBEDDING_DIM),
}

BACKEND_COLORS = {
    'tfidf':    '#4878CF',
    'minilm':   '#6ACC65',
    'mpnet':    '#D65F5F',
    'multi-qa': '#E89C3A',
    'openai':   '#B47CC7',
}


def run_backend(name):
    """Run one backend, return (embeddings_array, label_string)."""
    print(f'\n  Backend: {name}')
    if name not in BACKEND_DEFS:
        raise ValueError(f'Unknown backend: {name}. '
                         f'Choose from: {list(BACKEND_DEFS.keys())} or compare')
    try:
        emb, label = BACKEND_DEFS[name]()
        return emb, label
    except ImportError as e:
        print(f'  SKIPPED ({e})')
        return None, None
    except Exception as e:
        print(f'  ERROR ({e})')
        return None, None


def similarity_diagnostics(embeddings, label):
    """Print high-similarity pairs diagnostic."""
    sim    = cosine_similarity(embeddings)
    pairs  = [(ALL_ZONES[i], ALL_ZONES[j], sim[i, j])
              for i in range(len(ALL_ZONES))
              for j in range(i + 1, len(ALL_ZONES))
              if sim[i, j] > 0.90]
    pairs.sort(key=lambda x: -x[2])
    print(f'  [{label}] High-similarity pairs (>0.90): {len(pairs)}')
    for a, b, s in pairs[:8]:
        print(f'    {a:<10} ↔ {b:<10}  {s:.4f}')
    if len(pairs) > 8:
        print(f'    ... and {len(pairs) - 8} more')
    return len(pairs), pairs


def save_embeddings(embeddings, label, suffix=''):
    """Save embeddings DataFrame and return it."""
    emb_cols = [f'emb_{i:03d}' for i in range(embeddings.shape[1])]
    emb_df   = pd.DataFrame(embeddings, columns=emb_cols)
    emb_df.insert(0, 'zone_class', ALL_ZONES)
    emb_df.insert(1, 'context',    [ZONE_CONTEXT[z] for z in ALL_ZONES])
    fname = f'zone_embeddings{suffix}.csv'
    emb_df.to_csv(fname, index=False)
    print(f'  Saved: {fname}  ({len(emb_df)} zones × {len(emb_cols)} dims)')
    return emb_df, emb_cols


def classifier_comparison(emb_df, emb_cols, label):
    """Quick RF + GB comparison: baseline vs + embeddings. Returns RF acc."""
    trips   = pd.read_csv(TRIPS_PATH)
    zon_csv = pd.read_csv(ZON_CSV_PATH).dropna(subset=['ZONECLASS'])
    trips['active_travel'] = trips['mode_type'].isin([1, 2]).astype(int)

    area_col = ('ShapeSTAre_corrected' if 'ShapeSTAre_corrected' in zon_csv.columns
                else 'ShapeSTAre')

    emb_lookup = emb_df.set_index('zone_class')[emb_cols].to_dict('index')
    bg_rows = []
    for geoid, grp in zon_csv.groupby('GEOID'):
        total = grp[area_col].sum()
        vec   = np.zeros(len(emb_cols))
        cov   = 0.0
        for _, row in grp.iterrows():
            z = row['ZONECLASS']
            if z in emb_lookup:
                w    = row[area_col] / total
                vec += np.array(list(emb_lookup[z].values())) * w
                cov += w
        if cov > 0:
            bg_rows.append({'GEOID': geoid, **dict(zip(emb_cols, vec))})

    bg_emb = pd.DataFrame(bg_rows)
    df     = trips.merge(bg_emb, left_on='o_bg', right_on='GEOID', how='left')

    BASE = ['age', 'gender', 'education', 'income_detailed', 'num_veh',
            'hhsize', 'trip_distance', 'trip_duration', 'd_purpose_category_imputed']
    EMB  = emb_cols

    y    = df['active_travel'].values
    imp  = SimpleImputer(strategy='median')
    Xb   = imp.fit_transform(df[BASE].values.astype(float))
    Xa   = imp.fit_transform(df[BASE + EMB].values.astype(float))

    idx_tr, idx_te = train_test_split(np.arange(len(y)), test_size=0.2,
                                       random_state=16, stratify=y)
    sc = StandardScaler()
    Xb_tr = sc.fit_transform(Xb[idx_tr]); Xb_te = sc.transform(Xb[idx_te])
    sc2 = StandardScaler()
    Xa_tr = sc2.fit_transform(Xa[idx_tr]); Xa_te = sc2.transform(Xa[idx_te])
    y_tr, y_te = y[idx_tr], y[idx_te]

    results = {}
    for name, clf_proto in [
        ('Random Forest',     RandomForestClassifier(random_state=16)),
        ('Gradient Boosting', GradientBoostingClassifier(random_state=16)),
        ('Logistic Reg.',     LogisticRegression(random_state=16, max_iter=1000)),
    ]:
        cb = clone(clf_proto); ca = clone(clf_proto)
        cb.fit(Xb_tr, y_tr); ca.fit(Xa_tr, y_tr)
        acc_b = accuracy_score(y_te, cb.predict(Xb_te))
        acc_a = accuracy_score(y_te, ca.predict(Xa_te))
        ll_b  = log_loss(y_te, cb.predict_proba(Xb_te))
        ll_a  = log_loss(y_te, ca.predict_proba(Xa_te))
        results[name] = dict(acc_b=acc_b, acc_a=acc_a, ll_b=ll_b, ll_a=ll_a)
        print(f'    {name:<22} Base={acc_b:.2%} (LL={ll_b:.4f}) '
              f'→ +Emb={acc_a:.2%} (LL={ll_a:.4f}) '
              f'Δ={acc_a - acc_b:+.2%}')

    return results['Random Forest']['acc_a']


# ─────────────────────────────────────────────────────────────
# MAIN — single backend or comparison mode
# ─────────────────────────────────────────────────────────────
print('\n' + '=' * 60)
print('STAGE 2 — CREATING EMBEDDINGS')
print(f'Backend: {BACKEND}')
print('=' * 60)

if BACKEND == 'compare':
    # Run all available backends, collect results
    backend_results = {}
    for bname in ['tfidf', 'minilm', 'mpnet', 'multi-qa', 'openai']:
        emb, lbl = run_backend(bname)
        if emb is None:
            continue
        n_pairs, _ = similarity_diagnostics(emb, bname)
        emb_df, emb_cols = save_embeddings(emb, lbl, f'_{bname}')
        print(f'  Running classifier comparison [{bname}]...')
        rf_acc = classifier_comparison(emb_df, emb_cols, lbl)
        backend_results[bname] = {
            'label': lbl, 'n_pairs': n_pairs,
            'rf_acc': rf_acc, 'embeddings': emb
        }

    # Summary comparison table
    print('\n\n── Backend Comparison Summary ──')
    print(f'  {"Backend":<12} {"High-sim pairs (>0.90)":>23} {"RF Acc (+Emb)":>15}')
    print('  ' + '─' * 53)
    for bname, res in backend_results.items():
        print(f'  {bname:<12} {res["n_pairs"]:>23} {res["rf_acc"]:>14.2%}')

    # Auto-select best backend by RF classification accuracy on residential/
    # non-residential discrimination. This is the default zone_embeddings.csv
    # used by downstream DML pipeline.
    if backend_results:
        best = max(backend_results, key=lambda b: backend_results[b]['rf_acc'])
        print(f'\n  Best backend by RF accuracy: {best}')
        emb_df_best, emb_cols_best = save_embeddings(
            backend_results[best]['embeddings'],
            backend_results[best]['label'], '')
        print(f'  Saved best embeddings as zone_embeddings.csv')

    # Comparison figure: similarity pairs vs RF accuracy
    if len(backend_results) > 1:
        fig_cmp, (ax_l, ax_r) = plt.subplots(1, 2, figsize=(12, 5))
        fig_cmp.suptitle('Backend Comparison: Corpus Similarity vs Classifier Performance',
                          fontweight='bold')
        names  = list(backend_results.keys())
        pairs  = [backend_results[b]['n_pairs'] for b in names]
        accs   = [backend_results[b]['rf_acc'] * 100 for b in names]
        colors = [BACKEND_COLORS.get(b, '#888888') for b in names]

        ax_l.bar(names, pairs, color=colors, alpha=0.85)
        ax_l.set_ylabel('High-Similarity Pairs (>0.90)')
        ax_l.set_title('Corpus Similarity Collapse\n(lower = more distinct embeddings)')
        ax_l.grid(axis='y', alpha=0.3)
        for i, v in enumerate(pairs):
            ax_l.text(i, v + 0.3, str(v), ha='center', va='bottom', fontsize=9)

        ax_r.bar(names, accs, color=colors, alpha=0.85)
        ax_r.set_ylabel('RF Test Accuracy (%) — Baseline + Embeddings')
        ax_r.set_title('Classifier Performance\n(higher = more predictive embeddings)')
        ax_r.set_ylim(88, 95)
        ax_r.grid(axis='y', alpha=0.3)
        for i, v in enumerate(accs):
            ax_r.text(i, v + 0.05, f'{v:.2f}%', ha='center', va='bottom', fontsize=9)

        plt.tight_layout()
        plt.savefig('fig_backend_comparison.png', dpi=150, bbox_inches='tight')
        plt.close()
        print('Saved: fig_backend_comparison.png')

else:
    # Single backend run
    emb, label = run_backend(BACKEND)
    if emb is None:
        sys.exit(1)

    similarity_diagnostics(emb, BACKEND)
    emb_df, emb_cols = save_embeddings(emb, label)
    # Also save a backend-labelled copy
    save_embeddings(emb, label, f'_{BACKEND}')

    # 2D visualisation
    pca2   = PCA(n_components=2, random_state=RANDOM_STATE)
    viz    = pca2.fit_transform(emb)
    ctx_colors = {'Residential': '#66BB6A',
                  'Non-Residential': '#FFA726', 'Other/PD': '#BDBDBD'}
    fig_e, ax_e = plt.subplots(figsize=(11, 8))
    ax_e.set_facecolor('#FAFAFA')
    ax_e.grid(alpha=0.3, linewidth=0.5)
    for ctx, color in ctx_colors.items():
        mask = np.array([ZONE_CONTEXT[z] == ctx for z in ALL_ZONES])
        ax_e.scatter(viz[mask, 0], viz[mask, 1], c=color, s=90,
                     alpha=0.85, zorder=3, edgecolors='white',
                     linewidths=0.5, label=ctx)
    for i, zone in enumerate(ALL_ZONES):
        ax_e.annotate(zone, (viz[i, 0], viz[i, 1]), fontsize=5.5,
                      ha='center', va='bottom', xytext=(0, 4),
                      textcoords='offset points', alpha=0.85)
    ax_e.legend(fontsize=9)
    ax_e.set_xlabel(f'PC1 ({pca2.explained_variance_ratio_[0]:.1%} var)')
    ax_e.set_ylabel(f'PC2 ({pca2.explained_variance_ratio_[1]:.1%} var)')
    ax_e.set_title(f'Zoning Code Embedding Space (2D PCA)\n{label}',
                   fontweight='bold')
    plt.tight_layout()
    plt.savefig('fig_embedding_space_v3.png', dpi=150, bbox_inches='tight')
    plt.close()
    print('Saved: fig_embedding_space_v3.png')

    print(f'\nRunning Stage 3 classifier comparison [{BACKEND}]...')
    classifier_comparison(emb_df, emb_cols, label)

print('\n✓ Pipeline v3 complete.')

In [ ]:
#the final file names are df (origin and destination in the county area),destination county, and origin county. df is the only one with geometry